# 06 Mz Alignment

**Purpose.** Check mass-axis dispersion and bin width across the cohort.

**Thesis reference.** §4.2 Data Characterisation

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

## Configuration

In [ ]:
PROFILE_DIR       = "profiles"
LABELS_FILE       = "labels/labels_main_any_ckd.json"
N_REFERENCE_PEAKS = 40      # strongest ubiquitous peaks to track
WINDOW_DA         = 0.30    # centroid search half-width around each peak
MIN_PRESENCE      = 0.95    # peak must be detectable in this fraction of patients
OUT_FILE          = "mz_alignment_results.json"


## Code

In [ ]:
import json
from pathlib import Path

import numpy as np
from scipy.signal import find_peaks
from scipy.stats import spearmanr, mannwhitneyu


def pid_num(pid):
    return int("".join(c for c in pid if c.isdigit()))


def main():
    prof = Path(PROFILE_DIR)
    F = np.load(prof / "mz_profiles.npy")
    mz = np.load(prof / "mz_centres.npy")
    with open(prof / "profile_pids.json") as fh:
        pids = json.load(fh)

    labels = None
    if Path(LABELS_FILE).exists():
        with open(LABELS_FILE) as fh:
            labels = json.load(fh).get("labels")
        keep = [i for i, p in enumerate(pids) if p in labels]
        F, pids = F[keep], [pids[i] for i in keep]

    ids = np.array([pid_num(p) for p in pids])
    res = mz[1] - mz[0]
    print(f"Patients: {len(pids)}   bins: {F.shape[1]:,}   "
          f"resolution: {res:.4f} Da\n")

    # 1. reference peaks from the mean profile
    mean_prof = F.mean(axis=0)
    height = np.percentile(mean_prof[mean_prof > 0], 99)
    peaks, props = find_peaks(mean_prof, height=height,
                              distance=int(round(1.0 / res)))
    if len(peaks) == 0:
        raise SystemExit("No peaks found; check the profiles.")

    order = np.argsort(props["peak_heights"])[::-1]
    peaks = peaks[order][:N_REFERENCE_PEAKS * 3]

    half = max(1, int(round(WINDOW_DA / res)))
    keep_peaks = []
    for pk in peaks:
        lo, hi = max(0, pk - half), min(F.shape[1], pk + half + 1)
        present = (F[:, lo:hi].sum(axis=1) > 0).mean()
        if present >= MIN_PRESENCE:
            keep_peaks.append(pk)
        if len(keep_peaks) >= N_REFERENCE_PEAKS:
            break

    if len(keep_peaks) < 5:
        raise SystemExit(f"Only {len(keep_peaks)} ubiquitous peaks found; "
                         "lower MIN_PRESENCE or widen WINDOW_DA.")

    print(f"Reference peaks tracked: {len(keep_peaks)}")
    print(f"  m/z range: {mz[min(keep_peaks)]:.2f} -- {mz[max(keep_peaks)]:.2f}\n")

    # 2. per-patient centroid of each reference peak
    n_pat = len(pids)
    cent = np.full((n_pat, len(keep_peaks)), np.nan)
    for j, pk in enumerate(keep_peaks):
        lo, hi = max(0, pk - half), min(F.shape[1], pk + half + 1)
        w = F[:, lo:hi]
        tot = w.sum(axis=1)
        ok = tot > 0
        cent[ok, j] = (w[ok] * mz[lo:hi]).sum(axis=1) / tot[ok]

    dev = cent - np.nanmedian(cent, axis=0, keepdims=True)
    patient_shift = np.nanmedian(dev, axis=1)      # Da, per patient

    # 3. does the shift trend with patient ID?
    good = np.isfinite(patient_shift)
    rho, p = spearmanr(ids[good], patient_shift[good])
    span = (np.nanpercentile(patient_shift, 97.5)
            - np.nanpercentile(patient_shift, 2.5))
    sd = np.nanstd(patient_shift)

    print("Per-patient m/z shift (median deviation across reference peaks)")
    print(f"  std dev              : {sd:.5f} Da  ({sd / res:.2f} bins)")
    print(f"  95% span             : {span:.5f} Da  ({span / res:.2f} bins)")
    print(f"  max |shift|          : {np.nanmax(np.abs(patient_shift)):.5f} Da")
    print(f"  trend with patient ID: rho = {rho:+.3f}, p = {p:.2e}\n")

    # early versus late thirds
    o = np.argsort(ids)
    third = len(o) // 3
    early = np.nanmedian(patient_shift[o[:third]])
    late = np.nanmedian(patient_shift[o[-third:]])
    print(f"  median shift, earliest third: {early:+.5f} Da")
    print(f"  median shift, latest third  : {late:+.5f} Da")
    print(f"  difference                  : {late - early:+.5f} Da "
          f"({abs(late - early) / res:.2f} bins)\n")

    # 4. verdict
    worst = max(span, abs(late - early))
    bins = worst / res
    if bins < 0.5:
        verdict = "STABLE"
        print("VERDICT: m/z axis is STABLE.")
        print("  Peak positions do not move appreciably across the cohort, so")
        print("  the centre-of-mass trend reflects compositional variation")
        print("  rather than calibration drift.")
        print("  Proceed with region discovery at 0.02 Da unchanged.")
    elif bins < 2.5:
        verdict = "MILD"
        print("VERDICT: MILD drift, comparable to one or two bins.")
        print("  Tolerable, but it will blur narrow features.")
        print("  Either coarsen the resolution to 0.05 Da, or widen the")
        print("  region half-width in the region-discovery step.")
    else:
        verdict = "SUBSTANTIAL"
        print("VERDICT: SUBSTANTIAL drift, several bins wide.")
        print("  Bin k does not correspond to the same mass in all patients,")
        print("  which undermines fine-resolution region discovery.")
        print("  Align profiles first, or work at 0.1 Da, and report this.")

    # 5. does the shift relate to class?
    if labels is not None:
        y = np.array([labels[p] for p in pids])
        a = patient_shift[(y == 1) & good]
        b = patient_shift[(y == 0) & good]
        if len(a) and len(b):
            u, pv = mannwhitneyu(a, b)
            auc = u / (len(a) * len(b))
            print(f"\nShift as a class predictor: AUC "
                  f"{max(auc, 1 - auc):.4f} (p = {pv:.3f})")
            if pv < 0.01:
                print("  WARNING: the shift itself separates the classes.")
                print("  Alignment is then not optional; report it.")
            else:
                print("  Shift does not separate the classes. Good.")

    with open(OUT_FILE, "w") as fh:
        json.dump({
            "n_patients":         int(n_pat),
            "resolution_da":      float(res),
            "n_reference_peaks":  len(keep_peaks),
            "reference_peak_mz":  [float(mz[p]) for p in keep_peaks],
            "shift_std_da":       float(sd),
            "shift_span95_da":    float(span),
            "shift_bins":         float(bins),
            "rho_with_id":        float(rho),
            "p_with_id":          float(p),
            "early_late_diff_da": float(late - early),
            "verdict":            verdict,
        }, fh, indent=2)
    print(f"\nWritten to {OUT_FILE}")


main()


## Run

In [ ]:
main()

In [1]:
PROFILE_DIR       = "profiles"
LABELS_FILE       = "labels/labels_main_any_ckd.json"
N_REFERENCE_PEAKS = 40      # strongest ubiquitous peaks to track
WINDOW_DA         = 0.30    # centroid search half-width around each peak
MIN_PRESENCE      = 0.95    # peak must be detectable in this fraction of patients
OUT_FILE          = "mz_alignment_results.json"


In [2]:
import json
from pathlib import Path

import numpy as np
from scipy.signal import find_peaks
from scipy.stats import spearmanr, mannwhitneyu


def pid_num(pid):
    return int("".join(c for c in pid if c.isdigit()))


def main():
    prof = Path(PROFILE_DIR)
    F = np.load(prof / "mz_profiles.npy")
    mz = np.load(prof / "mz_centres.npy")
    with open(prof / "profile_pids.json") as fh:
        pids = json.load(fh)

    labels = None
    if Path(LABELS_FILE).exists():
        with open(LABELS_FILE) as fh:
            labels = json.load(fh).get("labels")
        keep = [i for i, p in enumerate(pids) if p in labels]
        F, pids = F[keep], [pids[i] for i in keep]

    ids = np.array([pid_num(p) for p in pids])
    res = mz[1] - mz[0]
    print(f"Patients: {len(pids)}   bins: {F.shape[1]:,}   "
          f"resolution: {res:.4f} Da\n")

    # 1. reference peaks from the mean profile
    mean_prof = F.mean(axis=0)
    height = np.percentile(mean_prof[mean_prof > 0], 99)
    peaks, props = find_peaks(mean_prof, height=height,
                              distance=int(round(1.0 / res)))
    if len(peaks) == 0:
        raise SystemExit("No peaks found; check the profiles.")

    order = np.argsort(props["peak_heights"])[::-1]
    peaks = peaks[order][:N_REFERENCE_PEAKS * 3]

    half = max(1, int(round(WINDOW_DA / res)))
    keep_peaks = []
    for pk in peaks:
        lo, hi = max(0, pk - half), min(F.shape[1], pk + half + 1)
        present = (F[:, lo:hi].sum(axis=1) > 0).mean()
        if present >= MIN_PRESENCE:
            keep_peaks.append(pk)
        if len(keep_peaks) >= N_REFERENCE_PEAKS:
            break

    if len(keep_peaks) < 5:
        raise SystemExit(f"Only {len(keep_peaks)} ubiquitous peaks found; "
                         "lower MIN_PRESENCE or widen WINDOW_DA.")

    print(f"Reference peaks tracked: {len(keep_peaks)}")
    print(f"  m/z range: {mz[min(keep_peaks)]:.2f} -- {mz[max(keep_peaks)]:.2f}\n")

    # 2. per-patient centroid of each reference peak
    n_pat = len(pids)
    cent = np.full((n_pat, len(keep_peaks)), np.nan)
    for j, pk in enumerate(keep_peaks):
        lo, hi = max(0, pk - half), min(F.shape[1], pk + half + 1)
        w = F[:, lo:hi]
        tot = w.sum(axis=1)
        ok = tot > 0
        cent[ok, j] = (w[ok] * mz[lo:hi]).sum(axis=1) / tot[ok]

    dev = cent - np.nanmedian(cent, axis=0, keepdims=True)
    patient_shift = np.nanmedian(dev, axis=1)      # Da, per patient

    # 3. does the shift trend with patient ID?
    good = np.isfinite(patient_shift)
    rho, p = spearmanr(ids[good], patient_shift[good])
    span = (np.nanpercentile(patient_shift, 97.5)
            - np.nanpercentile(patient_shift, 2.5))
    sd = np.nanstd(patient_shift)

    print("Per-patient m/z shift (median deviation across reference peaks)")
    print(f"  std dev              : {sd:.5f} Da  ({sd / res:.2f} bins)")
    print(f"  95% span             : {span:.5f} Da  ({span / res:.2f} bins)")
    print(f"  max |shift|          : {np.nanmax(np.abs(patient_shift)):.5f} Da")
    print(f"  trend with patient ID: rho = {rho:+.3f}, p = {p:.2e}\n")

    # early versus late thirds
    o = np.argsort(ids)
    third = len(o) // 3
    early = np.nanmedian(patient_shift[o[:third]])
    late = np.nanmedian(patient_shift[o[-third:]])
    print(f"  median shift, earliest third: {early:+.5f} Da")
    print(f"  median shift, latest third  : {late:+.5f} Da")
    print(f"  difference                  : {late - early:+.5f} Da "
          f"({abs(late - early) / res:.2f} bins)\n")

    # 4. verdict
    worst = max(span, abs(late - early))
    bins = worst / res
    if bins < 0.5:
        verdict = "STABLE"
        print("VERDICT: m/z axis is STABLE.")
        print("  Peak positions do not move appreciably across the cohort, so")
        print("  the centre-of-mass trend reflects compositional variation")
        print("  rather than calibration drift.")
        print("  Proceed with region discovery at 0.02 Da unchanged.")
    elif bins < 2.5:
        verdict = "MILD"
        print("VERDICT: MILD drift, comparable to one or two bins.")
        print("  Tolerable, but it will blur narrow features.")
        print("  Either coarsen the resolution to 0.05 Da, or widen the")
        print("  region half-width in the region-discovery step.")
    else:
        verdict = "SUBSTANTIAL"
        print("VERDICT: SUBSTANTIAL drift, several bins wide.")
        print("  Bin k does not correspond to the same mass in all patients,")
        print("  which undermines fine-resolution region discovery.")
        print("  Align profiles first, or work at 0.1 Da, and report this.")

    # 5. does the shift relate to class?
    if labels is not None:
        y = np.array([labels[p] for p in pids])
        a = patient_shift[(y == 1) & good]
        b = patient_shift[(y == 0) & good]
        if len(a) and len(b):
            u, pv = mannwhitneyu(a, b)
            auc = u / (len(a) * len(b))
            print(f"\nShift as a class predictor: AUC "
                  f"{max(auc, 1 - auc):.4f} (p = {pv:.3f})")
            if pv < 0.01:
                print("  WARNING: the shift itself separates the classes.")
                print("  Alignment is then not optional; report it.")
            else:
                print("  Shift does not separate the classes. Good.")

    with open(OUT_FILE, "w") as fh:
        json.dump({
            "n_patients":         int(n_pat),
            "resolution_da":      float(res),
            "n_reference_peaks":  len(keep_peaks),
            "reference_peak_mz":  [float(mz[p]) for p in keep_peaks],
            "shift_std_da":       float(sd),
            "shift_span95_da":    float(span),
            "shift_bins":         float(bins),
            "rho_with_id":        float(rho),
            "p_with_id":          float(p),
            "early_late_diff_da": float(late - early),
            "verdict":            verdict,
        }, fh, indent=2)
    print(f"\nWritten to {OUT_FILE}")


main()


In [3]:
main()


Patients: 303   bins: 80,000   resolution: 0.0200 Da

Reference peaks tracked: 40
  m/z range: 59.03 -- 811.59

Per-patient m/z shift (median deviation across reference peaks)
  std dev            : 0.00865 Da  (0.43 bins)
  95% span           : 0.03869 Da  (1.93 bins)
  max |shift|        : 0.03439 Da
  trend with patient ID: rho = -0.051, p = 3.81e-01

  median shift, earliest third: -0.00078 Da
  median shift, latest third  : +0.00053 Da
  difference                  : +0.00132 Da (0.07 bins)

VERDICT: MILD drift, comparable to one or two bins.
  Tolerable, but it will blur narrow features.
  -> Either coarsen MZ_RESOLUTION to 0.05 Da, or widen the
     region half-width in step 5 (half_width_da >= 0.1).

Shift as a class predictor: AUC 0.5650 (p = 0.078)
  Shift does not separate the classes. Good.

Written to mz_alignment_results.json
